In [6]:
import pandas as pd
from analysis.utility import set_column_awards

df = pd.concat([
    pd.read_csv('../data/boosted_localization_report.csv'),
    pd.read_csv('../data/union_localization_report.csv'),
])

In [7]:
# Load evaluations so we can also get resolution rate and average trajectory length
from pathlib import Path
from analysis.models.openhands import Evaluation

system_dir = Path("/Users/calvin/all-hands/data/artificial-localization-recall/boosted")

system_evaluations: dict[str, Evaluation] = {}
for system in system_dir.iterdir():
    if not system.is_dir():
        continue
    evaluation = Evaluation.from_filepath(str(system))
    system_evaluations[system.name] = evaluation

In [8]:
rows = []
for system, evaluation in system_evaluations.items():
    for prediction in evaluation.output:
        try:
            rows.append({
                "system": system,
                "instance_id": prediction.instance_id,
                "resolved": evaluation.is_resolved(prediction.instance_id),
                "iterations": len(prediction.history) if prediction.history else None,
            })
        except Exception as e:
            print(e)
            continue

extra_df = pd.DataFrame(rows)
extra_df = extra_df.groupby('system').agg({
    "resolved": "mean",
    "iterations": "mean",
})

In [12]:
grouped_df = df.groupby('system').agg({
    "file_match": "mean",
    "file_precision": "mean",
    "function_match": "mean",
    "function_precision": "mean",
    "class_match": "mean",
    "class_precision": "mean",
})

localization_df = pd.merge(grouped_df, extra_df, on='system', how='left')

styled_df = localization_df.copy()

for column, descending in [
    ("resolved", True),
    ("iterations", False),
    ("file_match", True),
    ("file_precision", True),
    ("function_match", True),
    ("function_precision", True),
    ("class_match", True),
    ("class_precision", True),
]:
    set_column_awards(styled_df, column, descending)

styled_df

,file_match,file_precision,function_match,function_precision,class_match,class_precision,resolved,iterations
system,,,,,,,,
bare-localization,0.68,0.71,0.58,0.58,0.66,0.58,🥈 0.52,95.62
file-localization,0.76,0.76,0.57,0.56,0.73,0.61,🥉 0.50,🥉 84.10
function-localization,0.71,0.75,0.59,0.54,0.76,0.62,🥈 0.52,93.22
gt-localization,🥉 0.78,🥈 0.78,0.66,0.66,🥉 0.78,0.69,0.48,84.68
jl-localization,🥈 0.80,0.76,🥉 0.68,🥈 0.67,🥈 0.80,🥉 0.70,0.46,87.94
jp-gt-localization,0.74,🥇 0.79,0.56,0.58,0.72,0.62,🥉 0.50,91.16
jp-jl-localization,🥉 0.78,0.77,🥈 0.70,🥇 0.70,🥈 0.80,🥇 0.73,🥈 0.52,🥈 80.20
jp-no-localization,0.74,0.75,0.52,0.56,0.70,0.60,0.46,96.52
no-localization,0.69,🥉 0.78,0.61,🥉 0.66,0.71,🥈 0.72,🥈 0.52,96.00


In [14]:
# Test correlation between the *_match/*_precision columns and the resolved/iterations columns
from itertools import product

rows = []

feature_columns = ["file_match", "file_precision", "function_match", "function_precision", "class_match", "class_precision"]
output_columns = ["resolved", "iterations"]

for feature, output in product(feature_columns, output_columns):
    correlation = localization_df[feature].corr(localization_df[output])
    rows.append({
        "feature": feature,
        "output": output,
        "correlation": correlation,
    })

correlation_df = pd.DataFrame(rows)
correlation_df

,feature,output,correlation
0,file_match,resolved,-0.534843
1,file_match,iterations,-0.667333
2,file_precision,resolved,-0.072762
3,file_precision,iterations,-0.414658
4,function_match,resolved,0.023669
5,function_match,iterations,-0.563648
6,function_precision,resolved,-0.060740
7,function_precision,iterations,-0.407565
8,class_match,resolved,-0.266569
9,class_match,iterations,-0.590294
